In [28]:
import json
import os
from typing import Annotated, Dict, List, Literal, Optional, TypedDict

from bs4 import BeautifulSoup
from langchain import hub
from langchain.agents import AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain.tools import tool
from langchain_community.chat_models import ChatOllama
from langchain_community.tools import DuckDuckGoSearchResults, DuckDuckGoSearchRun
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool as core_tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_ollama import ChatOllama as OllamaCore
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import ToolNode
import requests



### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [29]:
model_name = "llama4:16x17b"
temperature = 0.2

llm = ChatOllama(
    model=model_name,
    temperature=temperature,
)

prompt = ChatPromptTemplate.from_template(
    "Answer in Ukrainian. Topic: '{topic}'. Give: 1) definition; 2) key benefits; 3) current research. Keep it concise, max 200 characters."
)

chain = prompt | llm | StrOutputParser()
response = chain.invoke({"topic": "Квантові обчислення"})
print(response)


Квантові обчислення - це новий рівень обчислювальної потужності, що використовує квантові явища. 
1) Визначення: технологія обробки інформації на основі квантової механіки.
2) Ключові переваги: швидкість, безпека, ефективність.
3) Поточні дослідження: розробка квантових процесорів, алгоритмів та застосувань.


### Summary

In this task, a **local Ollama model (llama4:16x17b)** was utilized through LangChain to generate a concise Ukrainian response about *quantum computing*.

The selection of a local model was guided by the following considerations:
- **Cost efficiency** – avoidance of external API calls, thereby reducing unnecessary expenses.
- **Privacy & NDA compliance** – execution is performed entirely on the local machine, ensuring that sensitive or project-related data is not transmitted to third-party servers.

This configuration demonstrates how LLM tools can be integrated into workflows while preserving **full control of data** and enabling usage in environments where security and confidentiality are of paramount importance.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [30]:
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI",
]

for t in topics:
    print(f"=== {t} ===")
    r = chain.invoke({"topic": t})
    print(r)
    print()

=== Баєсівські методи в машинному навчанні ===
Баєсівські методи в машинному навчанні ґрунтуються на теоремі Байєса. 

1. **Визначення**: Використання ймовірнісних моделей для передбачень.
2. **Ключові переваги**: Гнучкість, надійність та інтерпретація результатів.
3. **Поточні дослідження**: Застосування в глибокому навчанні та обробці природної мови.

=== Трансформери в машинному навчанні ===
Трансформери в машинному навчанні - це тип архітектури нейронних мереж. 

1. Визначення: використовують механізм уваги для обробки послідовностей даних.
2. Ключові переваги: ефективність у обробці довгих послідовностей, паралелізація обчислень.
3. Поточні дослідження: застосування у природному обробленні мови, комп'ютерному зорі.

=== Explainable AI ===
Пояснювана ШІ (Explainable AI) - це підхід до розробки штучного інтелекту, який забезпечує зрозумілість та інтерпретованість рішень моделі.

1. Визначення: Пояснювана ШІ надає інформацію про те, як модель приймає рішення.
2. Ключові переваги: під



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [31]:
!pip install -q langchain_community duckduckgo_search

In [32]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

display(search.invoke("Obama's first name?"))

"11 вер. 2025 р. — Barack Obama ; Kwame Raoul · Barack Hu ssein Obama II. (1961-08-04) August 4, 1961 (age 64) Honolulu, Hawaii, U.S.. If Hussein had been Obama's first name as opposed to middle name, do you still think he would have become President? r/Presidents icon. 15 квіт. 2025 р. — Stories tagged with Barack Obama . President Barack Hussein Obama 's father's name was Barack Hussein Obama. He was named after his father. Hussein, Obama's middle name, is a very common Arabic ... Obama received a number of nicknames prior to and during his presidency. His longstanding nickname “Barry” comes from abbreviating his first name ."

In [33]:
web_search = DuckDuckGoSearchResults(
    name="web_search",
    description="Search the web and return a JSON list of results with titles, snippets, and links."
)

agent = initialize_agent(
    tools=[web_search],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
)

topic = "Штучний інтелект"
prompt_text = (
    "You are a research assistant. Always answer in Ukrainian. "
    "Call the tool 'web_search' exactly once to retrieve results for the topic below. "
    "After receiving the Observation, immediately stop searching and produce the final answer. "
    "When you have 5 publications, stop. Do not issue any further Actions. "
    "From the returned JSON results/snippets, extract or infer exactly 5 recent peer-reviewed publications. "
    "For each item return: Title, Authors (write 'Автори не вказані' if missing), Year (if visible), and a one-sentence summary. "
    "Return a Ukrainian Markdown list of exactly 5 items: **Title** — Authors — Year — Short description. "
    f"\n\nTopic: {topic}"
)

result = agent.invoke({"input": prompt_text})
print(result["output"])




> Entering new AgentExecutor chain...
Thought: I will start by searching for recent peer-reviewed publications on the topic of штучний інтелект (artificial intelligence).

Action:
```json
{
  "action": "web_search",
  "action_input": "Штучний інтелект site:scholar.google.com"
}
```
Observation: snippet: Інститут проблем штучного інтелекту МОН і НАН України - Cited by 531 - Штучний інтелект - кібернетика - інформатика - богослов'я - теологія, title: Анатолій Шевченко - Google Scholar, link: https://scholar.google.com/citations?user=1tHelTYAAAAJ, snippet: Ректор, Національний університет - 3 587 цитувань - штучний інтелект - аналіз даних - машинне навчання, title: Наталя Шаховська - Google Академія, link: https://scholar.google.com/citations?user=GfRgzs4AAAAJ&hl=uk, snippet: Taras Shevchenko National University of Kyiv - 134 цитирования - штучний інтелект - прийняття рішень в умовах невизначеності - слабко с, title: Oleh Suprun, Олег Супрун - Академия Google, link: https://scholar.goog



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [34]:
@tool("web_parse", return_direct=False, description="Download a web page and extract the main readable text. Input: URL. Output: short block with TITLE, first ~1200 chars of TEXT, and URL.")
def web_parse(url: str, timeout: int = 20) -> str:
    """Fetch the given URL and return cleaned main text for LLM analysis."""
    r = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    html = r.text
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.string.strip() if soup.title and soup.title.string else ""
    text = " ".join(BeautifulSoup(html, "html.parser").get_text("\n").split())
    return f"TITLE: {title}\nTEXT: {text[:1200]}...\nURL: {url}"

search_tool = DuckDuckGoSearchResults()
search_tool.name = "web_search"
search_tool.description = (
    "Search the web for recent, authoritative sources about weather in Brazil, "
    "global demand, and macroeconomics. Returns title, URL, snippet, date if available."
)

python_repl_tool = PythonREPLTool()
python_repl_tool.name = "python_calc"
python_repl_tool.description = (
    "Execute short Python calculations for baselines (CAGR, scenario multipliers), "
    "simple stats, or quick data transforms."
)
TOOLS = [search_tool, web_parse, python_repl_tool]
OPENAI_TOOLS = [convert_to_openai_tool(t) for t in TOOLS]

model_name = os.getenv("LLM_MODEL", "llama4:16x17b")
temperature = float(os.getenv("LLM_TEMPERATURE", "0.2"))
llm = ChatOllama(model=model_name, temperature=temperature)

SYSTEM_PROMPT = (
    "You are BizAssist, a cautious but proactive business-analytics agent."
    "You can search the web, fetch pages, and run Python computations."
    "Given a textual description of recent sales and a request to forecast next-year exports/sales, you must:"
    "- Identify drivers (inflation, demand, weather)."
    "- Use web tools to pull recent macro/industry/weather info (sources & dates)."
    "- Use Python to build a simple, explainable baseline (CAGR or scenario multipliers)."
    "- Present a point forecast and a plausible range, with assumptions and citations."
    "- If data are insufficient, say so clearly and request what’s missing."
    "Constraints: Prefer authoritative sources; include publication dates; avoid hallucinating numbers; show formulas used."
    "Return ALL outputs strictly in Ukrainian."
)

REFLECTION_PROMPT = (
    "You are a reviewer. Read the full conversation and produce:"
    "1) 3-6 concrete critiques to improve the final answer."
    "2) Up to 3 essential follow-ups if truly necessary."
    "3) A short revision plan that the agent could apply in one more pass."
    "Return ALL outputs strictly in Ukrainian."
)

class GraphState(TypedDict):
    messages: Annotated[list, "Additive message state"]
    reflection: Optional[str]
    done: bool

def agent_node(state: GraphState) -> GraphState:
    messages = state["messages"]
    resp = llm.invoke(messages, tools=OPENAI_TOOLS, tool_choice="auto")
    out_messages = messages + [resp]
    return {**state, "messages": out_messages}


def reflect_node(state: GraphState) -> GraphState:
    messages = state["messages"]
    critique = llm.invoke([
        ("system", "You are a strict quality reviewer for business analytics outputs."),
        ("human", REFLECTION_PROMPT + "---\nFULL MESSAGES:" + " ".join([m.content if hasattr(m, 'content') else str(m) for m in messages]))
    ])
    return {**state, "reflection": critique.content}


def router(state: GraphState) -> Literal["tools", "reflect", "end"]:
    messages = state["messages"]
    last = messages[-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    if state.get("reflection"):
        return "end"
    return "reflect"

workflow = StateGraph(GraphState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(TOOLS))
workflow.add_node("reflect", reflect_node)
workflow.set_entry_point("agent")
workflow.add_edge("tools", "agent")
workflow.add_conditional_edges("agent", router, {"tools": "tools", "reflect": "reflect", "end": END})
workflow.add_edge("reflect", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

user_task = (
    "Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. "
    "Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації."
)
ENTRANCE = [("system", SYSTEM_PROMPT), ("human", user_task)]

if __name__ == "__main__":
    final_state = app.invoke(
        {"messages": ENTRANCE, "reflection": None, "done": False},
        config={"configurable": {"thread_id": "1"}}
    )
    for m in final_state["messages"][::-1]:
        if isinstance(m, AIMessage) and not m.tool_calls:
            print(m.content)
            break
    print("\n\nREFLECTION:\n")
    print(final_state.get("reflection", "<no reflection>") or "<no reflection>")


Для оцінки експорту апельсинів у 2025 році розглянемо кілька факторів, що впливають на цей процес: динаміку попередніх років, погодні умови в Бразилії та світові тенденції попиту на апельсини.

### 1. Історичні дані про експорт

Історичні дані про експорт апельсинів з Бразилії такі:
- 2021: 200 тонн
- 2022: 190 тонн
- 2023: 210 тонн
- 2024 (дані на кінець року не доступні, але відомо, що експорт становить 220 тонн на певний момент): припустимо, що ці дані будуть близькими до офіційних на кінець року.

### 2. Погодні умови в Бразилії

Бразилія є одним із найбільших виробників апельсинів у світі. Погодні умови, такі як посуха, заморозки чи надмірні опади, можуть суттєво вплинути на врожайність. На жаль, без конкретних даних про погодні умови в Бразилії за останні роки та їхній вплив на виробництво апельсинів, важко робити точні прогнози.

### 3. Світовий попит на апельсини

Попит на апельсини залежить від багатьох факторів, включаючи економічну ситуацію у країнах-імпортерах, зміну спожив